# 02c - ResNet50 (VGGFace2) Baseline Encoding Model

Uses `facenet-pytorch`'s `InceptionResnetV1` pretrained on VGGFace2 as
the face-specialized comparison model. Imports the wrapper from
`src/models/resnet_vggface2_wrapper.py`.

**Fix applied:** installing `facenet-pytorch` alone was pulling in a
version of Pillow that breaks (`cannot import name 'is_directory' from
'PIL._util'`). This notebook now installs `facenet-pytorch` together
with a pinned, compatible Pillow version in a single command, so pip
resolves them together instead of one silently breaking the other.

**Important:** after running the install cell, if this is a fresh
runtime, you may need to Restart the runtime once (Runtime > Restart
session) before running the diagnostic cell, for the correct Pillow
version to actually load. Do NOT re-run the install cell after
restarting - just continue to the diagnostic cell directly.

**Before trusting results**, run the diagnostic cell below and check
the printed layer names against `ASSUMED_LAYERS` in the wrapper file.

In [ ]:
!git clone https://github.com/sossyh/ffa-dnn-ablation.git
%cd ffa-dnn-ablation

In [ ]:
# Installing facenet-pytorch together with a pinned Pillow version in
# one command lets pip resolve them jointly, avoiding the broken
# PIL._util.is_directory import error seen when Pillow gets upgraded
# separately afterward.
!pip install nilearn decord python-dotenv facenet-pytorch "Pillow>=10.0,<11" --quiet

## Restart check

Run this cell. If it prints an ImportError, go to Runtime > Restart
session now, then come back and re-run this cell only (not the install
cell above) to confirm the fix took effect.

In [ ]:
try:
    import PIL
    from PIL._util import is_directory
    print(f"Pillow version: {PIL.__version__}")
    print("Pillow import check passed. Safe to continue.")
except ImportError as e:
    print("Pillow import still broken:", e)
    print("Go to Runtime > Restart session now, then re-run this cell only.")

In [ ]:
import os
from google.colab import userdata

os.environ["DROPBOX_DATASET_LINK"] = userdata.get("DROPBOX_DATASET_LINK")

## Step 1: Diagnostic - print the model's real layer names

Run this first. Compare the printed names against `ASSUMED_LAYERS` in
`src/models/resnet_vggface2_wrapper.py` (currently: `repeat_1`,
`repeat_2`, `repeat_3`, `block8`, `avgpool_1a`).

In [ ]:
from facenet_pytorch import InceptionResnetV1

_diagnostic_model = InceptionResnetV1(pretrained="vggface2")

print("Top-level named modules:\n")
for name, module in _diagnostic_model.named_children():
    print(name, "->", type(module).__name__)

del _diagnostic_model

## Step 2: Load the wrapper from src/

If any layer is missing, this prints a clear warning rather than
failing silently. Fix `ASSUMED_LAYERS` in the wrapper file if needed,
then re-run this cell after pushing/pulling the change.

In [ ]:
import sys
sys.path.append(".")

from src.models.resnet_vggface2_wrapper import ResNetVGGFace2Wrapper

model_wrapper = ResNetVGGFace2Wrapper()
print("\nActive layers for feature extraction:", model_wrapper.layers)

## Step 3: Load fMRI data and video list

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.data_loading import (
    download_algonauts_data,
    load_all_rois,
    get_video_list,
    load_video_frames,
)
from src.encoding_model import train_and_evaluate_encoding_model, region_mean_accuracy

download_algonauts_data(data_dir="data/raw")

fmri_dir = "data/raw/participants_data_v2021/mini_track"
subject = "sub01"
roi_data = load_all_rois(fmri_dir, subject)

NUM_TRAIN_VIDEOS = roi_data["FFA"].shape[0]

video_dir = "data/raw/AlgonautsVideos268_All_30fpsmax"
video_paths = get_video_list(video_dir)[:NUM_TRAIN_VIDEOS]

assert len(video_paths) == NUM_TRAIN_VIDEOS, (
    f"Video count ({len(video_paths)}) does not match fMRI row count "
    f"({NUM_TRAIN_VIDEOS})."
)

print(f"Using {len(video_paths)} training videos")

## Step 4: Extract activations (cached separately from AlexNet)

In [ ]:
os.makedirs("data/processed", exist_ok=True)
activation_cache_path = "data/processed/resnet_vggface2_activations.npz"

need_to_extract = True

if os.path.exists(activation_cache_path):
    cached = np.load(activation_cache_path, allow_pickle=True)
    all_activations = cached["all_activations"].item()
    cached_num_videos = next(iter(all_activations.values())).shape[0]

    if cached_num_videos == len(video_paths):
        print("Cache matches. Using cached activations.")
        need_to_extract = False
    else:
        print("Cache is stale, re-extracting.")

if need_to_extract:
    print("Extracting ResNet50-VGGFace2 activations (this will take a while)...")
    all_activations = {layer: [] for layer in model_wrapper.layers}

    for video_path in tqdm(video_paths):
        frames = load_video_frames(video_path, num_frames=8)
        layer_acts = model_wrapper.extract(frames)
        for layer in model_wrapper.layers:
            all_activations[layer].append(layer_acts[layer])

    for layer in all_activations:
        all_activations[layer] = np.stack(all_activations[layer])

    np.savez(activation_cache_path, all_activations=all_activations)
    print("Saved activations to", activation_cache_path)

for layer, acts in all_activations.items():
    print(f"{layer}: {acts.shape}")
    assert acts.shape[0] == NUM_TRAIN_VIDEOS, f"Layer {layer} shape mismatch."

print("\nAll shapes confirmed.")

## Step 5: Full sweep - every layer x every region

Uses the same validated `train_and_evaluate_encoding_model` (RidgeCV,
per-voxel alpha, standardization inside each fold) as the AlexNet
baseline, so results are directly comparable.

In [ ]:
os.makedirs("results/tables/resnet_vggface2", exist_ok=True)

results = []

for layer_name, X in all_activations.items():
    for region_name, Y in roi_data.items():
        voxel_scores = train_and_evaluate_encoding_model(X, Y)
        mean_acc = region_mean_accuracy(voxel_scores)
        results.append({"layer": layer_name, "region": region_name, "accuracy": mean_acc})
        print(f"{layer_name} -> {region_name}: {mean_acc:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv("results/tables/resnet_vggface2/baseline_accuracy.csv", index=False)
results_df.head()

## Step 6: Download results

In [ ]:
from google.colab import files

files.download("results/tables/resnet_vggface2/baseline_accuracy.csv")

Move the downloaded CSV to
`results/tables/resnet_vggface2/baseline_accuracy.csv` in your local
repo, then commit and push from VS Code.